# Convert the waste classifier to ONNX

Run these cells in Google Colab. Upload `waste_classifier.keras` and `class_names.json` when prompted. The notebook preserves the model's internal MobileNetV2 rescaling layer and uses real uploaded image data to compare Keras and ONNX Runtime outputs before downloading the ONNX file.

The project model was saved with Keras 3.13.2. Conversion tooling compatibility can vary by TensorFlow version. If conversion fails, keep the full error output and do not use an unverified ONNX file.

In [ ]:
%pip install -q tf2onnx onnx onnxruntime

## Upload the trained model and its class order

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = {"waste_classifier.keras", "class_names.json"}
missing_files = required_files - set(uploaded)
if missing_files:
    raise FileNotFoundError(f"Upload these files: {sorted(missing_files)}")

KERAS_PATH = "/content/waste_classifier.keras"
CLASS_NAMES_PATH = "/content/class_names.json"
ONNX_PATH = "/content/waste_classifier.onnx"
SAVED_MODEL_PATH = "/content/waste_saved_model"

## Load and verify the trained Keras model

The expected input is NHWC `(224, 224, 3)`. The Keras graph must contain the internal Rescaling layer, and its output count must match the six labels.

In [ ]:
import json
import shutil
import subprocess
import sys

import numpy as np
import tensorflow as tf

EXPECTED_CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as class_file:
    class_names = json.load(class_file)
assert class_names == EXPECTED_CLASSES, f"Unexpected class order: {class_names}"

model = tf.keras.models.load_model(KERAS_PATH, compile=False)
assert tuple(model.input_shape) == (None, 224, 224, 3), model.input_shape
assert model.output_shape[-1] == len(class_names), model.output_shape
rescaling_layers = [
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.Rescaling)
]
assert any(
    np.isclose(float(layer.scale), 1.0 / 127.5)
    and np.isclose(float(layer.offset), -1.0)
    for layer in rescaling_layers
), "Expected the model's internal MobileNetV2 Rescaling layer was not found."

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("Input:", model.input_shape, "Output:", model.output_shape)
print("Classes:", class_names)

## Export a serving signature and convert it

The serving signature accepts float32 NHWC pixels in the original 0–255 range. It calls the original model directly, so the model's internal rescaling remains part of the exported graph.

In [ ]:
@tf.function(
    input_signature=[
        tf.TensorSpec([None, 224, 224, 3], tf.float32, name="images")
    ]
)
def serving_function(images):
    return {"probabilities": model(images, training=False)}

shutil.rmtree(SAVED_MODEL_PATH, ignore_errors=True)
tf.saved_model.save(
    model,
    SAVED_MODEL_PATH,
    signatures={"serving_default": serving_function},
)

conversion_command = [
    sys.executable,
    "-m",
    "tf2onnx.convert",
    "--saved-model",
    SAVED_MODEL_PATH,
    "--signature_def",
    "serving_default",
    "--opset",
    "17",
    "--output",
    ONNX_PATH,
]
subprocess.run(conversion_command, check=True)
print("Converted ONNX model:", ONNX_PATH)

## Verify ONNX structure and compare against Keras

Upload a real image (for example, one of your dataset images). This cell applies the same EXIF correction, RGB conversion, and resize as the Flask app, then compares both models using unscaled float32 pixel values. The notebook will stop if the outputs do not agree within tolerance.

In [ ]:
import onnx
import onnxruntime as ort
from PIL import Image, ImageOps

onnx.checker.check_model(ONNX_PATH)
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"],
)
onnx_input = session.get_inputs()[0]
onnx_output = session.get_outputs()[0]
assert onnx_input.type == "tensor(float)", onnx_input.type
print("ONNX input:", onnx_input.name, onnx_input.shape, onnx_input.type)
print("ONNX output:", onnx_output.name, onnx_output.shape)

sample_upload = files.upload()
if not sample_upload:
    raise ValueError("Upload one real image to compare model outputs.")
sample_path = "/content/" + next(iter(sample_upload))
with Image.open(sample_path) as source_image:
    sample_image = ImageOps.exif_transpose(source_image).convert("RGB")
    sample_image = sample_image.resize((224, 224), Image.Resampling.BILINEAR)
image_batch = np.expand_dims(np.asarray(sample_image, dtype=np.float32), axis=0)

keras_output = np.asarray(model(image_batch, training=False))
onnx_result = session.run([onnx_output.name], {onnx_input.name: image_batch})[0]
np.testing.assert_allclose(onnx_result, keras_output, rtol=1e-3, atol=1e-4)

keras_scores = np.asarray(keras_output).reshape(-1)
best_index = int(np.argmax(keras_scores))
print("Keras/ONNX parity check: PASS")
print("Maximum absolute difference:", float(np.max(np.abs(onnx_result - keras_output))))
print("Real image prediction:", class_names[best_index])
print("Confidence:", round(float(keras_scores[best_index]) * 100, 2), "%")

## Download the verified ONNX file

Only run this cell after the ONNX structure check and real-image parity check pass. Copy the downloaded file into the project root beside `app.py`.

In [ ]:
files.download(ONNX_PATH)